# KADMON — Comparaison des techniques OT

**Kantorovich Anatomical Deviation Mapping of Neurotracts**

Ce notebook compare les six configurations compression × transport sur une seule paire de bundles. QuickBundles sert de baseline tractographique pour les méthodes K-Means et binning. Utilisez ensuite `2_Visualiser_Deplacements_3D.ipynb` pour explorer la méthode choisie dans FURY.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
BUNDLES_DIR = PROJECT_ROOT / "notebooks" / "bundles"

SOURCE_PATH = BUNDLES_DIR / "103818/nn_8mm/tractosearch_nn_8_0mm_all_CC_1_m_12mpts_rasmm.npy"
TARGET_PATH = BUNDLES_DIR / "433839/nn_8mm/tractosearch_nn_8_0mm_all_CC_1_m_12mpts_rasmm.npy"

In [2]:
import optuna
import pandas as pd
from optuna.trial import TrialState

from kadmon.comparison import ComparisonConfiguration, compare_configurations
from kadmon.compression import compress_binning, compress_quickbundles
from kadmon.io import load_bundle
from kadmon.selection import select_best_reid_trial

from kadmon.defaults import (
    BINNING_PARTIAL_BIN_SIZE,
    BINNING_PARTIAL_BINNING_NB,
    BINNING_PARTIAL_MASS,
    BINNING_PARTIAL_REPRESENTATIVE,
    BINNING_SINKHORN_BIN_SIZE,
    BINNING_SINKHORN_BINNING_NB,
    BINNING_SINKHORN_EPSILON,
    BINNING_SINKHORN_REPRESENTATIVE,
    KMEANS_PARTIAL_K,
    KMEANS_PARTIAL_MASS,
    KMEANS_SINKHORN_EPSILON,
    KMEANS_SINKHORN_K,
    QUICKBUNDLES_PARTIAL_MASS,
    QUICKBUNDLES_PARTIAL_THRESHOLD,
    QUICKBUNDLES_SINKHORN_EPSILON,
    QUICKBUNDLES_SINKHORN_THRESHOLD,
)

N_POINTS = 12
SEED = 42
TARGET_REPRESENTATIVES = 270
REPRESENTATIVE_TOLERANCE = 0.15
MAX_REPRESENTATIVES = 5000
OPTUNA_STUDY_PATH = PROJECT_ROOT / "notebooks/optuna/studies/optuna_reid.sqlite3"

Info: some functions in tractosearch.resampling are faster when 'numba' is installed


## Chargement

In [3]:
if not SOURCE_PATH.is_file():
    raise FileNotFoundError(f"Bundle source introuvable : {SOURCE_PATH}")
if not TARGET_PATH.is_file():
    raise FileNotFoundError(f"Bundle cible introuvable : {TARGET_PATH}")

bundle_a = load_bundle(SOURCE_PATH, n_points=N_POINTS)
bundle_b = load_bundle(TARGET_PATH, n_points=N_POINTS)
print(f"Source : {len(bundle_a)} streamlines")
print(f"Cible  : {len(bundle_b)} streamlines")

Source : 10954 streamlines
Cible  : 11497 streamlines


## Calcul des six techniques

> **Note méthodologique — budget de représentants comparable**
>
> Le premier bloc calcule les trials retenus pour la couverture et la résolution anatomiques. Un bloc séparé calcule ensuite l'analyse secondaire à budget de représentants approximativement contrôlé. On peut donc exécuter seulement le profil anatomique, ou poursuivre pour obtenir la comparaison côte à côte. L'égalité du budget ne peut pas être exacte : QuickBundles et le binning ont une cardinalité dépendante de la géométrie, tandis que K-Means impose directement son nombre de clusters.
>
> Les essais sont classés uniformément par exactitude Top-1 maximale, rang intra-identité moyen minimal, ratio intra/inter minimal, puis couverture transportée maximale. En mode budget comparable, cette même règle est appliquée uniquement aux essais situés dans la bande de représentants demandée.

In [4]:
sinkhorn = {"max_iter": 2000, "stop_threshold": 1e-6, "reject_threshold": 1e-5}
kmeans = {"max_iter": 300, "tol": 1e-4, "seed": SEED}

def configuration_from_parameters(name, parameters):
    compression, transport = name.rsplit("_", 1)
    if compression == "quickbundles":
        compression_parameters = {"threshold": parameters["threshold"]}
    elif compression == "kmeans":
        compression_parameters = {
            **kmeans,
            "n_clusters": min(parameters["n_clusters"], len(bundle_a), len(bundle_b)),
        }
    else:
        compression_parameters = {
            "bin_size": parameters["bin_size"],
            "binning_nb": parameters["binning_nb"],
            "method": parameters["method"],
            "n_points": N_POINTS,
        }
    transport_parameters = (
        {"mass": parameters["mass"]}
        if transport == "partial"
        else {**sinkhorn, "epsilon": parameters["epsilon"]}
    )
    return ComparisonConfiguration(compression, transport, compression_parameters, transport_parameters)

default_parameters = {
    "quickbundles_partial": {"threshold": QUICKBUNDLES_PARTIAL_THRESHOLD, "mass": QUICKBUNDLES_PARTIAL_MASS},
    "quickbundles_sinkhorn": {"threshold": QUICKBUNDLES_SINKHORN_THRESHOLD, "epsilon": QUICKBUNDLES_SINKHORN_EPSILON},
    "kmeans_partial": {"n_clusters": KMEANS_PARTIAL_K, "mass": KMEANS_PARTIAL_MASS},
    "kmeans_sinkhorn": {"n_clusters": KMEANS_SINKHORN_K, "epsilon": KMEANS_SINKHORN_EPSILON},
    "binning_partial": {"bin_size": BINNING_PARTIAL_BIN_SIZE, "binning_nb": BINNING_PARTIAL_BINNING_NB, "method": BINNING_PARTIAL_REPRESENTATIVE, "mass": BINNING_PARTIAL_MASS},
    "binning_sinkhorn": {"bin_size": BINNING_SINKHORN_BIN_SIZE, "binning_nb": BINNING_SINKHORN_BINNING_NB, "method": BINNING_SINKHORN_REPRESENTATIVE, "epsilon": BINNING_SINKHORN_EPSILON},
}

anatomical_configurations = {
    name: configuration_from_parameters(name, parameters)
    for name, parameters in default_parameters.items()
}
anatomical_results = compare_configurations(
    bundle_a, bundle_b, anatomical_configurations,
    max_representatives=MAX_REPRESENTATIVES,
)


## 1. Profil anatomique retenu

In [5]:
anatomical_metrics = pd.DataFrame([
    result["metrics"] for result in anatomical_results.values()
])
display(anatomical_metrics.sort_values("configuration").reset_index(drop=True))


,configuration,compression,transport,source_n_representatives,target_n_representatives,source_compression_ratio,target_compression_ratio,compression_time_s,mdf_time_s,transport_time_s,global_distance_mm,transport_objective_normalized,transported_mass,mean_mm,median_mm,max_mm,p95_mm,n_valid_representatives,n_untransported_representatives
0,binning+partial,binning,partial,55,52,0.005021,0.004523,1.183848,0.000492,0.001262,4.379683,0.104675,0.68,3.659289,3.538123,7.065357,5.773711,40,15
1,binning+sinkhorn,binning,sinkhorn,407,406,0.037155,0.035314,1.206587,0.049959,4.293846,7.310855,0.290421,1.00,6.021180,5.362900,15.015970,10.337414,407,0
2,kmeans+partial,kmeans,partial,40,40,0.003652,0.003479,2.236533,0.000186,0.001060,3.669652,0.086070,0.63,3.189204,3.232054,6.879091,4.183224,27,13
3,kmeans+sinkhorn,kmeans,sinkhorn,155,155,0.014150,0.013482,3.210087,0.007426,0.920005,6.996959,0.300141,1.00,6.033846,5.168241,14.958808,12.345562,155,0
4,quickbundles+partial,quickbundles,partial,91,93,0.008307,0.008089,0.116848,0.000778,0.002123,6.592056,0.211002,0.99,6.028555,5.409592,14.655257,13.323899,91,0
5,quickbundles+sinkhorn,quickbundles,sinkhorn,160,173,0.014607,0.015047,0.188490,0.011182,0.065642,8.536750,0.238650,1.00,6.232998,5.430074,13.218651,11.046148,160,0


## 2. Analyse secondaire à budget comparable

Cette cellule est indépendante du calcul anatomique précédent. Elle sélectionne les meilleurs trials RE-ID dans une bande de cardinalité commune, puis recalcule les six techniques.

In [6]:
selection_rows = []
selected_parameters = dict(default_parameters)
if not OPTUNA_STUDY_PATH.is_file():
    raise FileNotFoundError(f"Base Optuna introuvable : {OPTUNA_STUDY_PATH}")
storage = f"sqlite:///{OPTUNA_STUDY_PATH.resolve()}"
study_names = list(default_parameters)
for name in study_names:
    study = optuna.load_study(study_name=name, storage=storage)
    completed = [
        trial for trial in study.trials
        if trial.state == TrialState.COMPLETE
        and trial.value is not None
        and "mean_n_representatives" in trial.user_attrs
    ]
    candidates = [
        trial for trial in completed
        if abs(trial.user_attrs["mean_n_representatives"] - TARGET_REPRESENTATIVES)
        / TARGET_REPRESENTATIVES <= REPRESENTATIVE_TOLERANCE
    ]
    if not candidates:
        raise RuntimeError(f"Aucun essai {name} dans la bande de représentants demandée.")
    selected = select_best_reid_trial(candidates)
    selected_parameters[name] = dict(selected.params)
    selection_rows.append({
        "configuration": name, "trial": selected.number,
        "score_optuna": selected.value,
        "mean_n_representatives_optuna": selected.user_attrs["mean_n_representatives"],
        "écart_budget_%": 100 * (selected.user_attrs["mean_n_representatives"] / TARGET_REPRESENTATIVES - 1),
    })
# Les moyennes Optuna servent à trouver une zone commune, puis le budget
# KMEANS_SINKHORN_K-Means est recalé sur le nombre réellement produit pour cette paire.
qb = selected_parameters["quickbundles_partial"]
bp = selected_parameters["binning_partial"]
bs = selected_parameters["binning_sinkhorn"]
qb_counts = [len(compress_quickbundles(bundle, threshold=qb["threshold"])[0]) for bundle in (bundle_a, bundle_b)]
local_method_means = [sum(qb_counts) / len(qb_counts)]
for parameters in (bp, bs):
    counts = [len(compress_binning(
        bundle, bin_size=parameters["bin_size"],
        binning_nb=parameters["binning_nb"], method=parameters["method"],
        n_points=N_POINTS,
    )[0]) for bundle in (bundle_a, bundle_b)]
    local_method_means.append(sum(counts) / len(counts))
pair_target = int(round(pd.Series(local_method_means).median()))
for name in ("kmeans_partial", "kmeans_sinkhorn"):
    study = optuna.load_study(study_name=name, storage=storage)
    completed = [trial for trial in study.trials if trial.state == TrialState.COMPLETE and trial.value is not None]
    candidates = [
        trial for trial in completed
        if abs(trial.params["n_clusters"] - pair_target) / pair_target <= REPRESENTATIVE_TOLERANCE
    ]
    selected = select_best_reid_trial(candidates)
    selected_parameters[name] = dict(selected.params)
    selection_rows = [row for row in selection_rows if row["configuration"] != name]
    selection_rows.append({
        "configuration": name, "trial": selected.number,
        "score_optuna": selected.value,
        "mean_n_representatives_optuna": selected.user_attrs["mean_n_representatives"],
        "écart_budget_%": 100 * (selected.params["n_clusters"] / pair_target - 1),
    })
print(f"Budget comparable estimé pour cette paire : {pair_target} représentants")
display(pd.DataFrame(selection_rows).sort_values("configuration").reset_index(drop=True))


comparable_configurations = {
    name: configuration_from_parameters(name, parameters)
    for name, parameters in selected_parameters.items()
}
comparable_results = compare_configurations(
    bundle_a, bundle_b, comparable_configurations,
    max_representatives=MAX_REPRESENTATIVES,
)
comparable_metrics = pd.DataFrame([
    result["metrics"] for result in comparable_results.values()
])
display(comparable_metrics.sort_values("configuration").reset_index(drop=True))


Budget comparable estimé pour cette paire : 118 représentants


,configuration,trial,score_optuna,mean_n_representatives_optuna,écart_budget_%
0,binning_partial,27,0.487588,261.475806,-3.157109
1,binning_sinkhorn,19,0.798516,258.882698,-4.117519
2,kmeans_partial,43,0.356268,111.548387,-2.542373
3,kmeans_sinkhorn,10,0.485718,109.589443,-6.779661
4,quickbundles_partial,75,0.468097,296.122581,9.675030
5,quickbundles_sinkhorn,3,0.899574,293.953079,8.871511


,configuration,compression,transport,source_n_representatives,target_n_representatives,source_compression_ratio,target_compression_ratio,compression_time_s,mdf_time_s,transport_time_s,global_distance_mm,transport_objective_normalized,transported_mass,mean_mm,median_mm,max_mm,p95_mm,n_valid_representatives,n_untransported_representatives
0,binning+partial,binning,partial,138,97,0.012598,0.008437,1.190433,0.001326,0.002059,4.509821,0.100769,0.67,4.010807,3.677270,7.329886,6.771791,101,37
1,binning+sinkhorn,binning,sinkhorn,138,97,0.012598,0.008437,1.183328,0.004879,0.017978,9.886618,0.189857,1.00,6.875169,6.703892,14.411266,10.522139,138,0
2,kmeans+partial,kmeans,partial,115,115,0.010498,0.010003,3.380878,0.001277,0.002393,3.198043,0.061921,0.54,2.947464,2.990186,4.665625,3.971281,70,45
3,kmeans+sinkhorn,kmeans,sinkhorn,110,110,0.010042,0.009568,3.325879,0.003831,0.606809,7.442996,0.313762,1.00,5.980795,5.314906,13.896903,11.085839,110,0
4,quickbundles+partial,quickbundles,partial,91,93,0.008307,0.008089,0.120599,0.000798,0.002108,6.592056,0.211002,0.99,6.028555,5.409592,14.655257,13.323899,91,0
5,quickbundles+sinkhorn,quickbundles,sinkhorn,91,93,0.008307,0.008089,0.121652,0.002952,0.006326,12.215810,0.088496,1.00,8.404071,9.011975,14.026438,12.190389,91,0


## 3. Comparaison côte à côte

Le tableau contient exactement deux colonnes de valeurs — `Profil anatomique` et `Budget comparable`. Les techniques et les mesures sont placées dans l'index des lignes pour permettre une comparaison horizontale directe. Ce bloc requiert l'exécution des deux analyses précédentes.

In [8]:
import optuna

STUDY_PATH = PROJECT_ROOT / "notebooks" / "optuna" / "studies" / "optuna_reid.sqlite3"
REID_TRIALS = {
    "quickbundles_partial": ("quickbundles_partial", 75),
    "quickbundles_sinkhorn": ("quickbundles_sinkhorn", 114),
    "kmeans_partial": ("kmeans_partial", 44),
    "kmeans_sinkhorn": ("kmeans_sinkhorn", 130),
    "binning_partial": ("binning_partial", 55),
    "binning_sinkhorn": ("binning_sinkhorn", 374),
}
TECHNIQUE_NAMES = {
    "quickbundles_partial": "QuickBundles + Partial OT",
    "quickbundles_sinkhorn": "QuickBundles + Sinkhorn",
    "kmeans_partial": "KMEANS_SINKHORN_K-Means + Partial OT",
    "kmeans_sinkhorn": "KMEANS_SINKHORN_K-Means + Sinkhorn",
    "binning_partial": "Binning + Partial OT",
    "binning_sinkhorn": "Binning + Sinkhorn",
}
def format_parameters(parameters):
    return "; ".join(
        f"{('k' if key == 'n_clusters' else 'representative' if key == 'method' else key)}={value:.8g}"
        if isinstance(value, float) else
        f"{('k' if key == 'n_clusters' else 'representative' if key == 'method' else key)}={value}"
        for key, value in parameters.items()
    )

def short_bundle_name(name):
    prefix = "tractosearch_nn_8_0mm_all_"
    return name.removeprefix(prefix).removesuffix("_m")

if not STUDY_PATH.is_file():
    raise FileNotFoundError(f"Base Optuna Re-ID introuvable : {STUDY_PATH}")
storage = f"sqlite:///{STUDY_PATH}"
optuna.logging.set_verbosity(optuna.logging.WARNING)
trial_numbers_by_analysis = {
    "Profil anatomique": REID_TRIALS,
    "Budget comparable": {
        row["configuration"]: (row["configuration"], int(row["trial"]))
        for row in selection_rows
    },
}

summary_rows = []
results_by_analysis = {
    "Profil anatomique": anatomical_results,
    "Budget comparable": comparable_results,
}
parameters_by_analysis = {
    "Profil anatomique": default_parameters,
    "Budget comparable": selected_parameters,
}

for analysis_name, analysis_results in results_by_analysis.items():
    selections = trial_numbers_by_analysis[analysis_name]
    for key in TECHNIQUE_NAMES:
        study_name, trial_number = selections[key]
        study = optuna.load_study(study_name=study_name, storage=storage)
        trial = next((item for item in study.trials if item.number == trial_number), None)
        if trial is None or trial.state != optuna.trial.TrialState.COMPLETE:
            raise RuntimeError(f"Essai Re-ID indisponible : {study_name} #{trial_number}")
        reid = trial.user_attrs
        result_metrics = analysis_results[key]["metrics"]
        total_time_s = sum(
            float(result_metrics[name])
            for name in ("compression_time_s", "mdf_time_s", "transport_time_s")
        )
        recorded_failures = reid.get("reid_failed_bundles")
        failures = (
            [short_bundle_name(name) for name in recorded_failures]
            if recorded_failures is not None else None
        )
        summary_rows.append({
            "Analyse": analysis_name,
            "Technique": TECHNIQUE_NAMES[key],
            "Trial Optuna": trial_number,
            "Re-ID score": f"{reid['intra_identity_top1_accuracy']:.1%}",
            "Bundles non Re-ID": (
                "Non enregistré" if failures is None
                else ", ".join(failures) if failures else "Aucun"
            ),
            "Paramètres": format_parameters(parameters_by_analysis[analysis_name][key]),
            "Déplacement moyen": f"{result_metrics['mean_mm']:.3f} mm",
            "Nombre de représentants": (
                f"{result_metrics['source_n_representatives']} source / "
                f"{result_metrics['target_n_representatives']} cible"
            ),
            "Temps d'exécution": f"{total_time_s:.3f} s",
        })

technique_summary = pd.DataFrame(summary_rows)
comparison_table = (
    technique_summary
    .melt(id_vars=["Analyse", "Technique"], var_name="Mesure", value_name="Valeur")
    .pivot(index=["Technique", "Mesure"], columns="Analyse", values="Valeur")
    .reindex(columns=["Profil anatomique", "Budget comparable"])
)
comparison_table.columns.name = None
display(comparison_table)


Profil anatomique  \
Technique                            Mesure                                                                       
Binning + Partial OT                 Bundles non Re-ID                                                    CST_L   
                                     Déplacement moyen                                                 3.659 mm   
                                     Nombre de représentants                               55 source / 52 cible   
                                     Paramètres               bin_size=16; binning_nb=2; representative=mean...   
                                     Re-ID score                                                          96.8%   
                                     Temps d'exécution                                                  1.186 s   
                                     Trial Optuna                                                            55   
Binning + Sinkhorn                   Bundles non Re-ID                                                    CST_L   
                                     Déplacement moyen                                                 6.021 mm   
                                     Nombre de représentants                             407 source / 406 cible   
                                     Paramètres               bin_size=8; binning_nb=2; representative=mean;...   
                                     Re-ID score                                                          96.8%   
                                     Temps d'exécution                                                  5.550 s   
                                     Trial Optuna                                                           374   
KMEANS_SINKHORN_K-Means + Partial OT Bundles non Re-ID                                           Non enregistré   
                                     Déplacement moyen                                                 3.189 mm   
                                     Nombre de représentants                               40 source / 40 cible   
                                     Paramètres                                                 k=40; mass=0.63   
                                     Re-ID score                                                         100.0%   
                                     Temps d'exécution                                                  2.238 s   
                                     Trial Optuna                                                            44   
KMEANS_SINKHORN_K-Means + Sinkhorn   Bundles non Re-ID                                            CST_L, IFOF_R   
                                     Déplacement moyen                                                 6.034 mm   
                                     Nombre de représentants                             155 source / 155 cible   
                                     Paramètres                                      k=155; epsilon=0.017736981   
                                     Re-ID score                                                          93.5%   
                                     Temps d'exécution                                                  4.138 s   
                                     Trial Optuna                                                           130   
QuickBundles + Partial OT            Bundles non Re-ID                                            CST_L, IFOF_R   
                                     Déplacement moyen                                                 6.029 mm   
                                     Nombre de représentants                               91 source / 93 cible   
                                     Paramètres                                          threshold=7; mass=0.99   
                                     Re-ID score                                                          93.5%   
                                     Temps d'exécution               

## 4. Analyse des résultats

- **K-Means + Partial OT** obtient la meilleure RE-ID (**100 %**).
- **QuickBundles** est nettement le plus rapide.
- **Binning + Sinkhorn** atteint **96,8 %** RE-ID, mais utilise le plus de représentants et coûte le plus cher.
- À budget comparable (~118 représentants), les méthodes Partial OT restent stables, tandis que Binning et QuickBundles + Sinkhorn baissent à **83,9 %**.